# Compare Fisheye Transform Methods

## 1. OpenCV Fisheye Transform

In [ ]:
import os
from pathlib import Path

import cv2
import numpy as np

current_dir = Path(os.getcwd())
root_dir    = current_dir.parents[0]
data_dir    = root_dir / "data" / "ftt"
image_file  = data_dir / "0000305_00001_d_0000213.jpg"
output_file = image_file.parent / f"{image_file.stem}_cv2.jpg"

img    = cv2.imread(str(image_file))
h0, w0 = img.shape[:2]
size0  = min(h0, w0)

scale     = 2
img_large = cv2.resize(img, None, fx=scale, fy=scale, interpolation=cv2.INTER_LANCZOS4)

h, w   = img_large.shape[:2]
size   = min(h, w)
img    = img[:size, :size]  # Crop to square

cx, cy = size / 2, size / 2
f      = size / np.pi  # For 180 deg full circle
K      = np.array([[f, 0, cx], [0, f, cy], [0, 0, 1]], dtype=np.float32)
D      = np.array([-0.11, -0.11, 0.0, 0.0], dtype=np.float32)  # Equidistant model; adjust for other projections

# Maps
xd, yd      = np.meshgrid(np.arange(size), np.arange(size))
dist_pts    = np.stack((xd, yd), axis=-1).reshape(-1, 1, 2).astype(np.float32)
undist_norm = cv2.fisheye.undistortPoints(dist_pts, K, D)

xu = undist_norm[:, 0, 0] * f + cx
yu = undist_norm[:, 0, 1] * f + cy

map_x = xu.reshape(size, size).astype(np.float32)
map_y = yu.reshape(size, size).astype(np.float32)

# Remap
fisheye_img = cv2.remap(img_large, map_x, map_y, cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT)
fisheye_img = fisheye_img[
    size // 2 - size0 // 2: size // 2 + size0 // 2,
    size // 2 - size0 // 2: size // 2 + size0 // 2
]

cv2.imwrite(str(output_file), fisheye_img)